# Spindle — All 12 Domains

Spindle ships 12 production-ready domains, each with statistically calibrated distributions sourced from authoritative industry data (NRF, CMS, CDC, BLS, HCUP, and more).

| Domain | Tables | Focus |
|---|---|---|
| retail | 9 | Customers, orders, products, returns |
| healthcare | 9 | Patients, encounters, diagnoses, claims |
| financial | 8 | Accounts, transactions, loans, fraud |
| supply_chain | 9 | Warehouses, POs, inventory, logistics |
| iot | 8 | Devices, sensors, alerts, maintenance |
| hr | 8 | Employees, departments, compensation |
| insurance | 8 | Policies, claims, underwriting |
| marketing | 8 | Campaigns, leads, conversions |
| education | 8 | Students, courses, grades, financial aid |
| real_estate | 8 | Properties, listings, transactions |
| manufacturing | 8 | Production, quality control, equipment |
| telecom | 8 | Subscribers, usage, billing, churn |

In [ ]:
%pip install sqllocks-spindle --quiet

In [ ]:
import time
import pandas as pd
from sqllocks_spindle import (
    Spindle,
    RetailDomain, HealthcareDomain, FinancialDomain, SupplyChainDomain,
    IoTDomain, HRDomain, InsuranceDomain, MarketingDomain,
    EducationDomain, RealEstateDomain, ManufacturingDomain, TelecomDomain,
)

spindle = Spindle()

DOMAINS = [
    ("retail",        RetailDomain()),
    ("healthcare",    HealthcareDomain()),
    ("financial",     FinancialDomain()),
    ("supply_chain",  SupplyChainDomain()),
    ("iot",           IoTDomain()),
    ("hr",            HRDomain()),
    ("insurance",     InsuranceDomain()),
    ("marketing",     MarketingDomain()),
    ("education",     EducationDomain()),
    ("real_estate",   RealEstateDomain()),
    ("manufacturing", ManufacturingDomain()),
    ("telecom",       TelecomDomain()),
]

## Generate all domains at `fabric_demo` scale

In [ ]:
results = {}
bench = []
t_total = time.time()

for name, domain in DOMAINS:
    t0 = time.time()
    r = spindle.generate(domain=domain, scale="fabric_demo", seed=42)
    elapsed = time.time() - t0
    results[name] = r
    total_rows = sum(len(r[t]) for t in r.tables)
    errors = r.verify_integrity()
    status = "PASS" if not errors else f"FAIL ({len(errors)})"
    bench.append({"domain": name, "tables": len(r.tables), "rows": total_rows,
                  "elapsed_s": round(elapsed, 2), "integrity": status})
    print(f"{name:<16} {len(r.tables):>2} tables  {total_rows:>7,} rows  {elapsed:.2f}s  {status}")

print(f"\nTotal: {time.time() - t_total:.1f}s")

In [ ]:
pd.DataFrame(bench)

## Retail — Orders

In [ ]:
results["retail"]["order"].head()

## Healthcare — Diagnoses (ICD-10 codes)

In [ ]:
results["healthcare"]["diagnosis"].head()

## Financial — Fraud detection

Spindle's financial domain includes a `is_fraud` flag calibrated to realistic fraud rates.

In [ ]:
txn = results["financial"]["transaction"]
print(f"Fraud rate: {txn['is_fraud'].mean():.2%}")
txn.head()

## IoT — Sensor readings

In [ ]:
results["iot"]["sensor_reading"].head()

## Telecom — Churn risk

In [ ]:
churn = results["telecom"]["churn_indicator"]
print(f"Churn rate: {churn['churned'].mean():.1%}")
print("\nChurn risk distribution:")
print(churn["churn_risk_level"].value_counts())

## HR — Salary by department

In [ ]:
emp = results["hr"]["employee"]
emp.groupby("department_id")["salary"].describe().round(0)

## Marketing — Lead funnel

In [ ]:
leads = results["marketing"]["lead"]
print("Lead status distribution:")
print(leads["status"].value_counts(normalize=True).mul(100).round(1).rename("pct %"))